# 05 — Exploratory Data Analysis


In [2]:
#Imports and data loading

from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

ROOT = Path.cwd().parent

DATA_FILE = ROOT / "artifacts" / "cleaned_data.parquet"
CHART_DIR = ROOT / "artifacts" / "charts"

CHART_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(DATA_FILE)

ID_COLUMNS = ["customer_id"]
TARGET = "purchased"

analysis_df = df.drop(
    columns=[column for column in ID_COLUMNS if column in df.columns]
)

numeric_columns = [
    column
    for column in analysis_df.select_dtypes(include=np.number).columns
    if column != TARGET
]

categorical_columns = (
    analysis_df.select_dtypes(exclude=np.number).columns.tolist()
)

print("Shape:", analysis_df.shape)
print("Numeric columns:", numeric_columns)
print("Categorical columns:", categorical_columns)

display(analysis_df.head())

Shape: (500, 6)
Numeric columns: ['age', 'income', 'visits', 'satisfaction']
Categorical columns: ['region']


,age,income,region,visits,satisfaction,purchased
0,58,47184,North,9,2.0,0
1,65,44868,North,19,2.7,0
2,23,75314,South,17,3.4,1
3,63,127974,South,15,3.4,1
4,66,59853,West,11,2.1,0


In [3]:
#Target Distribution


target_counts = (
    analysis_df[TARGET]
    .value_counts()
    .sort_index()
    .reset_index()
)

target_counts.columns = [TARGET, "count"]
target_counts["class"] = target_counts[TARGET].astype(str)

target_counts["percentage"] = (
    target_counts["count"]
    / target_counts["count"].sum()
    * 100
).round(2)

display(target_counts)

fig = px.bar(
    target_counts,
    x="class",
    y="count",
    color="class",
    text="percentage",
    title="Target Class Distribution",
    labels={
        "class": "Purchased",
        "count": "Number of customers",
    },
)

fig.update_traces(
    texttemplate="%{text:.2f}%",
    textposition="outside",
)

fig.update_layout(showlegend=False)
fig.show()

fig.write_html(CHART_DIR / "target_distribution.html")


,purchased,count,class,percentage
0,0,196,0,39.2
1,1,304,1,60.8


In [4]:
#Numerical Distributions

for column in numeric_columns:
    fig = px.histogram(
        analysis_df,
        x=column,
        color=analysis_df[TARGET].astype(str),
        marginal="box",
        nbins=30,
        barmode="overlay",
        opacity=0.65,
        title=f"Distribution of {column} by Purchase Status",
        labels={"color": "Purchased"},
    )

    fig.show()

    fig.write_html(
        CHART_DIR / f"{column}_distribution.html"
    )

In [5]:
#Box plot against the target

for column in numeric_columns:
    fig = px.box(
        analysis_df,
        x=analysis_df[TARGET].astype(str),
        y=column,
        color=analysis_df[TARGET].astype(str),
        points="outliers",
        title=f"{column} by Purchase Status",
        labels={
            "x": "Purchased",
            "color": "Purchased",
        },
    )

    fig.update_layout(showlegend=False)
    fig.show()

    fig.write_html(
        CHART_DIR / f"{column}_target_boxplot.html"
    )

In [6]:
#Correlation Heatmap


correlation_columns = numeric_columns + [TARGET]

pearson_correlation = analysis_df[
    correlation_columns
].corr(method="pearson")

fig = px.imshow(
    pearson_correlation,
    text_auto=".2f",
    color_continuous_scale="RdBu_r",
    zmin=-1,
    zmax=1,
    aspect="auto",
    title="Pearson Correlation Matrix",
)

fig.show()
fig.write_html(CHART_DIR / "pearson_correlation.html")

display(pearson_correlation.round(3))

,age,income,visits,satisfaction,purchased
age,1.000,0.006,-0.029,-0.087,-0.089
income,0.006,1.000,0.009,0.009,0.398
visits,-0.029,0.009,1.000,0.037,0.368
satisfaction,-0.087,0.009,0.037,1.000,0.330
purchased,-0.089,0.398,0.368,0.330,1.000


In [7]:
# Region vs purchase rate

region_summary = (
    analysis_df.groupby("region", as_index=False)
    .agg(
        customers=(TARGET, "size"),
        purchases=(TARGET, "sum"),
        purchase_rate=(TARGET, "mean"),
    )
)

region_summary["purchase_rate_percentage"] = (
    region_summary["purchase_rate"] * 100
).round(2)

display(region_summary)

fig = px.bar(
    region_summary,
    x="region",
    y="purchase_rate_percentage",
    color="region",
    text="purchase_rate_percentage",
    title="Purchase Rate by Region",
    labels={
        "purchase_rate_percentage": "Purchase rate (%)"
    },
)

fig.update_traces(
    texttemplate="%{text:.2f}%",
    textposition="outside",
)

fig.update_layout(showlegend=False)
fig.show()

fig.write_html(CHART_DIR / "region_purchase_rate.html")

,region,customers,purchases,purchase_rate,purchase_rate_percentage
0,East,127,74,0.582677,58.27
1,North,124,73,0.588710,58.87
2,South,115,71,0.617391,61.74
3,West,134,86,0.641791,64.18


In [8]:
# Income, visits and satisfaction

fig = px.scatter(
    analysis_df,
    x="income",
    y="visits",
    color=analysis_df[TARGET].astype(str),
    size="satisfaction",
    hover_data=["age", "region"],
    opacity=0.75,
    title="Income vs Visits, Sized by Satisfaction",
    labels={"color": "Purchased"},
)

fig.show()
fig.write_html(CHART_DIR / "income_visits_satisfaction.html")

In [9]:
#Automated EDA Insights

purchase_rates = analysis_df.groupby("region")[TARGET].mean()

highest_region = purchase_rates.idxmax()
lowest_region = purchase_rates.idxmin()

target_correlations = (
    analysis_df[correlation_columns]
    .corr(method="spearman")[TARGET]
    .drop(TARGET)
    .sort_values(key=abs, ascending=False)
)

strongest_feature = target_correlations.index[0]
strongest_correlation = target_correlations.iloc[0]

eda_insights = [
    (
        f"The dataset contains {len(analysis_df):,} records "
        f"and {analysis_df.shape[1]} analytical variables."
    ),
    (
        f"The purchase rate is "
        f"{analysis_df[TARGET].mean() * 100:.2f}%."
    ),
    (
        f"The numerical feature with the strongest Spearman "
        f"relationship with purchase is {strongest_feature} "
        f"({strongest_correlation:.3f})."
    ),
    (
        f"{highest_region} has the highest observed purchase "
        f"rate at {purchase_rates.max() * 100:.2f}%."
    ),
    (
        f"{lowest_region} has the lowest observed purchase "
        f"rate at {purchase_rates.min() * 100:.2f}%."
    ),
    (
        "The regional differences should not be treated as "
        "meaningful because the chi-square test was not significant."
    ),
]

for number, insight in enumerate(eda_insights, start=1):
    print(f"{number}. {insight}")

1. The dataset contains 500 records and 6 analytical variables.
2. The purchase rate is 60.80%.
3. The numerical feature with the strongest Spearman relationship with purchase is income (0.394).
4. West has the highest observed purchase rate at 64.18%.
5. East has the lowest observed purchase rate at 58.27%.
6. The regional differences should not be treated as meaningful because the chi-square test was not significant.


In [10]:
chart_files = sorted(CHART_DIR.glob("*.html"))

print(f"Charts created: {len(chart_files)}")

for chart_file in chart_files:
    print(chart_file.name)

Charts created: 12
age_distribution.html
age_target_boxplot.html
income_distribution.html
income_target_boxplot.html
income_visits_satisfaction.html
pearson_correlation.html
region_purchase_rate.html
satisfaction_distribution.html
satisfaction_target_boxplot.html
target_distribution.html
visits_distribution.html
visits_target_boxplot.html
